In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import StrEnum
from functools import cache
from pathlib import Path
from queue import deque
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import seaborn as sns
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor, plot_tree

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import DIVERGING_CMAP, configure_mpl
from ising import Ising, SymmetricIsing

RANDOM_SEED = 202606301

rng = np.random.default_rng(RANDOM_SEED)

np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")

schema = schema.post_index()

## Load data

In [ ]:
type ModelType = Literal["ising", "sym_ising"]


class InterventionStrength(StrEnum):
    NULL = "null"
    WEAK = ("weak",)
    MEDIUM = ("medium",)
    MODERATE = "moderate"
    STRONG = "strong"
    PERFECT = "perfect"

    def path(self, model_type: ModelType, data_dir: Path) -> Path:
        return data_dir / self.filename(model_type)

    def filename(self, model_type: ModelType) -> Path:
        return Path(f"{model_type}_{self.delta_str()}.npz")

    def delta(self) -> float:
        match self:
            case InterventionStrength.NULL:
                return 0.0
            case InterventionStrength.WEAK:
                return 0.5
            case InterventionStrength.MEDIUM:
                return 1.0
            case InterventionStrength.MODERATE:
                return 1.5
            case InterventionStrength.STRONG:
                return 2.5
            case InterventionStrength.PERFECT:
                return 8.0

    def delta_str(self) -> str:
        delta = self.delta()
        return str(delta).replace(".", "")


@cache
def load_intervention_results(
    strength: InterventionStrength,
    model_type: ModelType,
    data_dir: Path,
) -> np.lib.npyio.NpzFile:
    print(strength, model_type, data_dir)
    return np.load(strength.path(model_type, data_dir))


def load_intervention_effect(
    strength: InterventionStrength,
    data_dir: Path,
    measure_time: int,
    intervention_idx: int | None = None,
    target_idx: int | None = None,
    model_type: ModelType = "ising",
) -> npt.NDArray[np.int64]:
    # Load final state from intervention model
    intervention_outcome = load_intervention_results(
        strength,
        model_type,
        data_dir,
    )["measurements"][:, :, measure_time]

    # Load final state from no-intervention model
    null_outcome = load_intervention_results(
        InterventionStrength.NULL,
        model_type,
        data_dir,
    )["measurements"][:, :, measure_time]

    # Calculate effect as the counterfactual difference
    effect = intervention_outcome - null_outcome

    # Optionally filter down to a specific intervention column
    #   i.e., the belief/attitude which we influence
    if intervention_idx is not None:
        effect = effect[:, :, intervention_idx]

    # Optionally filter down to a specific target column
    #   i.e., the belief/attitude we ultimately want to change
    if target_idx is not None:
        effect = effect[..., target_idx]

    return effect


def load_effective_baseline_activation(
    strength: InterventionStrength,
    data_dir: Path,
    measure_time: int | None = None,
    intervention_idx: int | None = None,
    target_idx: int | None = None,
    model_type: ModelType = "ising",
) -> npt.NDArray[np.float64]:
    results = load_intervention_results(strength, model_type, data_dir)
    params = results["params"]
    S0 = results["Y"][:, :, 0]
    S = results["measurements"][:, :, :, intervention_idx]

    match model_type:
        case "ising":
            model_cls = Ising
        case "sym_ising":
            model_cls = SymmetricIsing
        case _:
            raise ValueError(f"Invalid model type: '{model_type}'")

    R, M, T, N = S.shape
    X = np.array([1.0])
    adj = np.ones((N, N), dtype=np.bool)

    if measure_time is not None:
        X = np.ones(M, dtype=np.float64)
        h_eff = np.empty((R, M, N), dtype=np.float64)

        for repeat in range(R):
            p = params[repeat]
            h = p[:N]
            j = p[N:].reshape((N, N))

            prevs = S0[repeat] if measure_time == 0 else S[repeat, :, measure_time - 1]
            h_eff[repeat] = model_cls.parallel_glauber_theta_batch(prevs, X, h, j, adj)

    else:
        X = np.ones(1, dtype=np.float64)
        h_eff = np.empty((R, M, T, N), dtype=np.float64)

        for repeat in range(R):
            p = params[repeat]
            h = p[:N]
            j = p[N:].reshape((N, N))

            for t in range(T):
                prevs = S0[repeat] if t == 0 else S[repeat, :, t - 1]
                h_eff[repeat, :, t] = model_cls.parallel_glauber_theta_batch(
                    prevs, X, h, j, adj
                )

    if target_idx is not None:
        h_eff = h_eff[..., target_idx]

    return h_eff

In [ ]:
null_res = load_intervention_results(InterventionStrength.NULL, "ising", DATA_PATH)
int_res_weak = load_intervention_results(InterventionStrength.WEAK, "ising", DATA_PATH)
int_res_strong = load_intervention_results(
    InterventionStrength.STRONG, "ising", DATA_PATH
)

null_measurements = null_res["measurements"][:, :, :, 0]
int_measurements_weak = int_res_weak["measurements"][:, :, :, 0]
int_measurements_strong = int_res_strong["measurements"][:, :, :, 0]


params_weak = int_res_weak["params"].copy()
params_weak[:, 0] += InterventionStrength.WEAK.delta()
params_strong = int_res_strong["params"].copy()
params_strong[:, 0] += InterventionStrength.STRONG.delta()

In [ ]:
null_res["measurements"][0, :, 5, 0].mean(axis=0)

In [ ]:
null_res["params"][0, :8]

In [ ]:
int_res_strong["measurements"][:, 0, 5, 0].mean(axis=0)

In [ ]:
print(params_weak[0, :8])

In [ ]:
str(null_res["model_type"])

Calculate activation probability at $t=5$. i.e., given previous state at $t=4$.

In [ ]:
def calc_activation_prob(prevs, h, j):
    adj = np.ones((h.size, h.size), dtype=np.int64)
    h_eff = Ising.parallel_glauber_theta_batch(
        prevs,
        np.ones(prevs.shape[0], dtype=np.float64),
        h,
        j,
        adj,
    )
    return np.exp(h_eff) / (2 * np.cosh(h_eff))


def calc_activation_probs(params, measurements, t):
    repeats, individuals, _, n = measurements.shape
    p = np.empty((repeats, individuals, n), dtype=np.float64)
    for r in range(repeats):
        h = params[r][:n]
        j = params[r][n:].reshape((n, n))
        p[r] = calc_activation_prob(measurements[r, :, t - 1], h, j)
        # for i in range(individuals):
        #     p[r,i] = calc_activation_prob(measurements[r, i, t-1], h, j)
    return p

In [ ]:
p_null = calc_activation_probs(null_res["params"], null_measurements, t=5)

In [ ]:
p_int_weak_old = calc_activation_probs(
    int_res_weak["params"], int_measurements_weak, t=5
)
p_int_strong_old = calc_activation_probs(
    int_res_strong["params"], int_measurements_strong, t=5
)

In [ ]:
p_int_weak_new = calc_activation_probs(params_weak, int_measurements_weak, t=5)
p_int_strong_new = calc_activation_probs(params_strong, int_measurements_strong, t=5)

In [ ]:
null_results = load_intervention_results(InterventionStrength.NULL, "ising", DATA_PATH)

labels = null_results["labels"]

# measurements: (repeat, individual, timestep, intervention, target)
S0 = null_results["measurements"][0, :, 0, 0]

# Y0: (individual, data timestep, spin)
X0 = null_results["Y0"][:, 1]

## Compare average intervention effect to ratio of probabilities

In [ ]:
for strength in InterventionStrength:
    if strength == InterventionStrength.NULL:
        continue

    effect = load_intervention_effect(
        strength,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )

    h_eff_null = load_effective_baseline_activation(
        InterventionStrength.NULL,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )

    h_eff_int = load_effective_baseline_activation(
        strength,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )

    avg_p_null = (np.exp(h_eff_null) / (2 * np.cosh(h_eff_null))).mean(axis=0)
    avg_p_int = (np.exp(h_eff_int) / (2 * np.cosh(h_eff_int))).mean(axis=0)
    avg_p_ratio = avg_p_int / avg_p_null
    avg_effect = effect.mean(axis=0)

    plt.scatter(avg_effect, avg_p_ratio, s=5, alpha=0.5, label=str(strength))
    plt.xlabel("Effect of intervention")
    plt.ylabel("Ratio of $P(S_i^5 = +1)$")
    plt.title(f"Intervention: {strength.delta()}")
plt.legend();

## Regressing average intervention effect on initial state

For each individual, calculate the average effect of intervention ($S_{i, \text{intervention}}^t - S_{i, \text{null}}^t$), and fit a linear regression model for this, taking the initial raw measurement as a predictor. Include pairwise polynomial features.

In [ ]:
def fit_lasso_cv(X, Y) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.float64], Any]:
    model = Pipeline([("poly", PolynomialFeatures()), ("lasso", LassoCV(cv=10))]).fit(
        X, Y
    )
    degree_1_features = model["lasso"].coef_[1:9]
    degree_2_features = np.zeros((8, 8), dtype=np.float64)
    degree_2_features[np.triu_indices_from(degree_2_features)] = model["lasso"].coef_[
        9:
    ]

    degree_1_features[abs(degree_1_features) < 1e-4] = 0
    degree_2_features[abs(degree_2_features) < 1e-4] = 0

    return degree_1_features, degree_2_features, model

In [ ]:
for intervention in (
    InterventionStrength.WEAK,
    InterventionStrength.MODERATE,
    InterventionStrength.STRONG,
):
    if intervention == InterventionStrength.NULL:
        continue

    # poly = PolynomialFeatures()
    # X = poly.fit_transform(X0)
    Y = load_intervention_effect(
        intervention, DATA_PATH, measure_time=5, intervention_idx=2, target_idx=7
    ).mean(axis=0)
    β1, β2, model = fit_lasso_cv(X0, Y)

    fig, axes = plt.subplots(
        nrows=2, figsize=(5.5, 6.5), constrained_layout=True, height_ratios=(1, 3)
    )
    sns.barplot(β1, ax=axes[0])
    axes[0].set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")

    axes[1].set_aspect("equal")
    mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
    sns.heatmap(
        β2[::-1],
        mask=mask,
        vmin=-0.05,
        vmax=0.05,
        center=0,
        cmap=DIVERGING_CMAP,
        annot=True,
        fmt=".2f",
        linewidth=0.5,
        cbar_kws=dict(shrink=0.65, aspect=25),
        ax=axes[1],
    )
    axes[1].set_yticks(np.arange(8) + 0.5, reversed(labels), rotation=0)
    axes[1].set_xticks(
        np.arange(8) + 0.5, labels, rotation=35, horizontalalignment="right"
    )

    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

    fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
    ax.set_aspect("equal")
    Y_pred = model.predict(X0)
    ax.scatter(
        Y,
        Y_pred,
        s=10,
    )
    ax.set_xlabel(r"$Y$")
    ax.set_ylabel(r"$\hat{Y}$")
    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

## Regressing average intervention effect on initial local fields

For each individual, calculate the average effect of intervention ($S_{i, \text{intervention}}^t - S_{i, \text{null}}^t$), and fit a linear regression model for this, taking the average initial _local fields_ as a predictor. Include pairwise polynomial features.

In [ ]:
h_eff = load_effective_baseline_activation(
    InterventionStrength.NULL,
    DATA_PATH,
    measure_time=0,
    intervention_idx=2,
).mean(axis=0)

for intervention in InterventionStrength:
    if intervention == InterventionStrength.NULL:
        continue

    poly = PolynomialFeatures()
    X = poly.fit_transform(h_eff)
    Y = load_intervention_effect(
        intervention,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)
    β1, β2 = fit_lasso_cv(X, Y)

    fig, axes = plt.subplots(
        nrows=2,
        figsize=(5.5, 6.5),
        constrained_layout=True,
        height_ratios=(1, 3),
    )
    sns.barplot(β1, ax=axes[0])
    axes[0].set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")

    axes[1].set_aspect("equal")
    mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
    sns.heatmap(
        β2[::-1],
        mask=mask,
        vmin=-0.05,
        vmax=0.05,
        center=0,
        cmap=DIVERGING_CMAP,
        annot=True,
        fmt=".2f",
        linewidth=0.5,
        cbar_kws=dict(shrink=0.65, aspect=25),
        ax=axes[1],
    )
    axes[1].set_yticks(
        np.arange(8) + 0.5,
        reversed(labels),
        rotation=0,
    )
    axes[1].set_xticks(
        np.arange(8) + 0.5,
        labels,
        rotation=35,
        horizontalalignment="right",
    )

    fig.suptitle(f"$\\delta = {intervention.delta()}$")

## Regressing ratio of final probabilities on initial activation probabilities

For each individual, calculate the average probability that the target spin is $+1$ at $t=5$, and the corresponding ratio with the average probability in the null model. Fit a linear regresson model to this, taking the average initial spin probabilities as predictors. 

In [ ]:
h_eff = load_effective_baseline_activation(
    InterventionStrength.NULL,
    DATA_PATH,
    measure_time=0,
    intervention_idx=2,
).mean(axis=0)

init_p = np.exp(h_eff) / (2 * np.cosh(h_eff))

for intervention in (
    InterventionStrength.WEAK,
    InterventionStrength.MODERATE,
    InterventionStrength.STRONG,
):
    if intervention == InterventionStrength.NULL:
        continue

    h_eff_final_null = load_effective_baseline_activation(
        InterventionStrength.NULL,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)
    h_eff_final_int = load_effective_baseline_activation(
        intervention,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)

    final_p_null = np.exp(h_eff_final_null) / (2 * np.cosh(h_eff_final_null))
    final_p_int = np.exp(h_eff_final_int) / (2 * np.cosh(h_eff_final_int))

    ratio = np.exp(np.log(final_p_int) - np.log(final_p_null))

    X = init_p
    Y = ratio
    model = LassoCV(cv=10).fit(X, Y)
    β1 = model.coef_

    fig, ax = plt.subplots(
        figsize=(5.5, 2.5),
        constrained_layout=True,
    )
    sns.barplot(β1, ax=ax)
    ax.set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")
    ax.set_ylim(-1.0, 0.2)
    ax.axhline(y=0, linestyle="dashed", linewidth=0.5, color="grey")

    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

## Regressing difference of final probabilities on initial state

For each individual, calculate the average probability that the target spin is $+1$ at $t=5$, and the corresponding ratio with the average probability in the null model. Fit a linear regresson model to this, taking the initial raw measurement as a predictor. Include pairwise polynomial features.

In [ ]:
for intervention in (
    InterventionStrength.WEAK,
    InterventionStrength.MODERATE,
    InterventionStrength.STRONG,
):
    if intervention == InterventionStrength.NULL:
        continue

    h_eff_final_null = load_effective_baseline_activation(
        InterventionStrength.NULL,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )
    h_eff_final_int = load_effective_baseline_activation(
        intervention,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )

    final_p_null = (np.exp(h_eff_final_null) / (2 * np.cosh(h_eff_final_null))).mean(
        axis=0
    )
    final_p_int = (np.exp(h_eff_final_int) / (2 * np.cosh(h_eff_final_int))).mean(
        axis=0
    )

    diff = final_p_int - final_p_null

    # poly = PolynomialFeatures()
    # X = poly.fit_transform(X0)
    Y = diff
    β1, β2, model = fit_lasso_cv(X0, Y)

    fig, axes = plt.subplots(
        nrows=2,
        figsize=(5.5, 6.5),
        constrained_layout=True,
        height_ratios=(1, 3),
    )
    sns.barplot(β1, ax=axes[0])
    axes[0].set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")

    axes[1].set_aspect("equal")
    mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
    vlim = np.max(abs(β2))
    sns.heatmap(
        β2[::-1],
        mask=mask,
        vmin=-vlim,
        vmax=vlim,
        center=0,
        cmap=DIVERGING_CMAP,
        annot=True,
        fmt=".2f",
        linewidth=0.5,
        cbar_kws=dict(shrink=0.65, aspect=25),
        ax=axes[1],
    )
    axes[1].set_yticks(
        np.arange(8) + 0.5,
        reversed(labels),
        rotation=0,
    )
    axes[1].set_xticks(
        np.arange(8) + 0.5,
        labels,
        rotation=35,
        horizontalalignment="right",
    )

    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

    fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
    ax.set_aspect("equal")
    Y_pred = model.predict(X0)
    ax.scatter(
        Y,
        Y_pred,
        s=10,
    )
    ax.set_xlabel(r"$Y$")
    ax.set_ylabel(r"$\hat{Y}$")
    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

## Shallow decision tree

In [ ]:
@dataclass
class Leaf:
    prediction: float | list[float]
    samples: int


@dataclass
class Node:
    feature_idx: int
    feature_name: str
    threshold: float
    samples: int
    children: tuple[Node | Leaf | None, Node | Leaf | None]


def make_tree_recursive(root_idx: int, clf_tree, labels: [list[str]]) -> Node | Leaf:
    left_child_idx = clf_tree.children_left[root_idx]
    right_child_idx = clf_tree.children_right[root_idx]

    match left_child_idx, right_child_idx:
        case (-1, -1):
            return Leaf(
                clf_tree.value[root_idx].item(), int(clf_tree.n_node_samples[root_idx])
            )
        case (_, -1):
            left_child = make_tree_recursive(left_child_idx, clf_tree, labels)
            right_child = None
        case (-1, _):
            left_child = None
            right_child = make_tree_recursive(right_child_idx, clf_tree, labels)
        case _:
            left_child = make_tree_recursive(left_child_idx, clf_tree, labels)
            right_child = make_tree_recursive(right_child_idx, clf_tree, labels)

    feature_idx = int(clf_tree.feature[root_idx])
    feature_name = str(labels[feature_idx])
    threshold = float(clf_tree.threshold[root_idx])
    samples = int(clf_tree.n_node_samples[root_idx])
    return Node(
        feature_idx, feature_name, threshold, samples, (left_child, right_child)
    )


def prune_tree(tree: Node | Leaf, le: float, ge: float) -> Node | Leaf | None:
    # If leaf and don't satisfy bounds, prune.
    if isinstance(tree, Leaf):
        if isinstance(tree.prediction, list):
            raise TypeError("Unexpected list prediction type.")
        if ge <= tree.prediction <= le:
            return tree
        else:
            return None

    # If node, first recurse on children...
    new_left_child = prune_tree(tree.children[0], le, ge)
    new_right_child = prune_tree(tree.children[1], le, ge)

    # Then if both children have been pruned, prune the whole node
    if new_left_child is None and new_right_child is None:
        return None

    # Otherwise reassign the children, update sample count, and return self
    tree.children = (new_left_child, new_right_child)
    tree.samples = 0
    for child in tree.children:
        if child is not None:
            tree.samples += child.samples

    return tree


def merge_on_coverings(tree: Node | Leaf | None) -> Node | Leaf:
    # If leaf, return self. If node, merge subtrees.
    match tree:
        case None | Leaf():
            return tree
        case Node(children=(c1, c2)):
            tree.children = (merge_on_coverings(c1), merge_on_coverings(c2))

    # Then (if node), if both children are leaves, replace current node be leaf
    left, right = tree.children
    if isinstance(left, Leaf) and isinstance(right, Leaf):
        left_predictions = (
            [left.prediction] if isinstance(left.prediction, float) else left.prediction
        )
        right_predictions = (
            [right.prediction]
            if isinstance(right.prediction, float)
            else right.prediction
        )
        return Leaf(
            prediction=sorted(left_predictions + right_predictions),
            samples=tree.samples,
        )

    # Otherwise, return self
    return tree


def extract_rule_tree(
    clf, labels: list[str], le: float | None = None, ge: float | None = None
):
    _le = le or np.inf
    _ge = ge or -np.inf

    clf_tree = clf.tree_

    # Construct full tree
    tree = make_tree_recursive(0, clf_tree, labels)

    # Prune subtrees which don't satisfy bounds
    tree = prune_tree(tree, _le, _ge)

    # Merge subtrees where whole feature range is covered
    tree = merge_on_coverings(tree)

    return tree


@dataclass
class Interval:
    lower: float = -1.0
    upper: float = 1.0

    def __hash__(self):
        return hash((self.lower, self.upper))


@dataclass
class RuleNode:
    feature_idx: int
    feature_name: str
    interval: Interval

    def __hash__(self):
        return hash((self.feature_idx, self.feature_name, self.interval))


def extract_rules_old(
    clf, labels: list[str], le: float | None = None, ge: float | None = None
):
    tree = extract_rule_tree(clf, labels, le, ge)

    # Create list of paths from root to leaves
    paths = []
    queue = deque()
    queue.append([(None, tree)])
    while queue:
        path = queue.popleft()
        _, head = path[-1]

        for pos, child in zip(("left", "right"), head.children, strict=True):
            new_path = path + [(pos, child)]
            if isinstance(child, Node):
                queue.append(new_path)
            elif isinstance(child, Leaf):
                paths.append(new_path)

    # Convert each path to a collection of rule nodes
    rules = []
    for path in paths:
        rule = []
        feature_idx = path[0][1].feature_idx
        feature_name = path[0][1].feature_name
        threshold = path[0][1].threshold
        for pos, node in path[1:]:
            interval = (
                Interval(upper=threshold)
                if pos == "left"
                else Interval(lower=threshold)
            )
            rule.append(RuleNode(feature_idx, feature_name, interval))
            if not isinstance(node, Node):
                rules.append(rule)
                break
            feature_idx = node.feature_idx
            feature_name = node.feature_name
            threshold = node.threshold

    return rules


def extract_rules(
    clf, labels: list[str], le: float | None = None, ge: float | None = None
):
    tree = extract_rule_tree(clf, labels, le, ge)

    if isinstance(tree, Leaf):
        return []

    # Create list of paths from root to leaves
    paths = []
    queue = deque()
    queue.append([(None, tree)])
    while queue:
        path = queue.popleft()
        _, head = path[-1]

        for pos, child in zip(("left", "right"), head.children, strict=True):
            new_path = path + [(pos, child)]
            if isinstance(child, Node):
                queue.append(new_path)
            elif isinstance(child, Leaf):
                paths.append(new_path)

    # Convert each path to a collection of rule nodes
    rules = []
    for path in paths:
        rule = dict()
        feature_idx = path[0][1].feature_idx
        feature_name = path[0][1].feature_name
        threshold = float(
            np.round(np.round(path[0][1].threshold / 0.5) * 0.5, decimals=1)
        )

        for pos, node in path[1:]:
            rule_node = rule.setdefault(
                feature_idx, RuleNode(feature_idx, feature_name, Interval())
            )
            if pos == "left":
                rule_node.interval.upper = min(rule_node.interval.upper, threshold)
            elif pos == "right":
                rule_node.interval.lower = max(rule_node.interval.lower, threshold)
            else:
                raise ValueError(f"Unexpected value for pos: {pos}.")

            if isinstance(node, Leaf):
                rules.append((node.samples, list(rule.values())))
                break
            elif not isinstance(node, Node):
                raise TypeError(f"Unexpected node type: {type(node)}")
            feature_idx = node.feature_idx
            feature_name = node.feature_name
            threshold = float(
                np.round(np.round(node.threshold / 0.5) * 0.5, decimals=1)
            )

    # Sort rules by feature idx and convert to tuple
    rules = [
        (freq, tuple(sorted(rule, key=lambda r: r.feature_idx))) for freq, rule in rules
    ]
    return rules

In [ ]:
from tqdm import trange

target_idx = 7

diffs = {}
for intervention_idx in trange(len(labels)):
    diffs[intervention_idx] = {}

    h_eff_final_null = load_effective_baseline_activation(
        InterventionStrength.NULL,
        DATA_PATH,
        measure_time=5,
        intervention_idx=intervention_idx,
        target_idx=target_idx,
    )
    final_p_null = np.exp(h_eff_final_null) / (2 * np.cosh(h_eff_final_null))

    for intervention in (
        InterventionStrength.WEAK,
        InterventionStrength.MODERATE,
        InterventionStrength.STRONG,
    ):
        h_eff_final_int = load_effective_baseline_activation(
            intervention,
            DATA_PATH,
            measure_time=5,
            intervention_idx=intervention_idx,
            target_idx=target_idx,
        )

        final_p_int = np.exp(h_eff_final_int) / (2 * np.cosh(h_eff_final_int))

        diff = (final_p_int - final_p_null).mean(axis=0)
        diffs[intervention_idx][intervention] = diff


```
[RuleNode(feature_idx=2, feature_name='CC worry', interval=Interval(lower=-1.0, upper=0.0)), RuleNode(feature_idx=5, feature_name='Politics', interval=Interval(lower=-1.0, upper=0.04129665344953537)), RuleNode(feature_idx=0, feature_name='Belief CC', interval=Interval(lower=-0.5, upper=1.0))]
[RuleNode(feature_idx=2, feature_name='CC worry', interval=Interval(lower=-1.0, upper=0.0)), RuleNode(feature_idx=7, feature_name='Climate Policy', interval=Interval(lower=-1.0, upper=0.03298698738217354)), RuleNode(feature_idx=5, feature_name='Politics', interval=Interval(lower=-1.0, upper=-0.055802229791879654))]
[RuleNode(feature_idx=2, feature_name='CC worry', interval=Interval(lower=-1.0, upper=0.0)), RuleNode(feature_idx=7, feature_name='Climate Policy', interval=Interval(lower=-1.0, upper=-0.005103766452521086)), RuleNode(feature_idx=1, feature_name='CC anthropogenic', interval=Interval(lower=-1.0, upper=0.0))]
```

```
[RuleNode(feature_idx=2, feature_name='CC worry', interval=Interval(lower=-1.0, upper=0.0)), RuleNode(feature_idx=5, feature_name='Politics', interval=Interval(lower=-1.0, upper=0.04129665344953537)), RuleNode(feature_idx=0, feature_name='Belief CC', interval=Interval(lower=-0.5, upper=1.0))]
[RuleNode(feature_idx=2, feature_name='CC worry', interval=Interval(lower=-1.0, upper=0.0)), RuleNode(feature_idx=7, feature_name='Climate Policy', interval=Interval(lower=-1.0, upper=0.03298698738217354)), RuleNode(feature_idx=5, feature_name='Politics', interval=Interval(lower=-1.0, upper=-0.055802229791879654))]
[RuleNode(feature_idx=2, feature_name='CC worry', interval=Interval(lower=-1.0, upper=0.0)), RuleNode(feature_idx=7, feature_name='Climate Policy', interval=Interval(lower=-1.0, upper=-0.005103766452521086)), RuleNode(feature_idx=1, feature_name='CC anthropogenic', interval=Interval(lower=-1.0, upper=0.0))]
```

In [ ]:
labels = np.asarray(
    [
        "CC Real",
        "CC Human",
        "CC Worry",
        "CC Others Worry",
        "W. Worry",
        "Politics",
        "CC Impact",
        "CC Policy",
    ]
)

In [ ]:
# ANNOTATIONS = {
#     (-1.0, -0.5): "Very low",
#     (-1.0, 0): "Low",
#     (-1.0, 0.5): "~Very high",
#     (-1.0, 1.0): "Any",
#     (-0.5, 0.0): "Slight low",
#     (-0.5, 0.5): "Neutral",
#     (-0.5, 1.0): "~Very low",
#     (0.0, 0.0): "Very neutral",
#     (0.0, 0.5): "Slight high",
#     (0.0, 1.0): "High",
#     (0.5, 1.0): "Very high",
# }

ANNOTATIONS = {
    (-1.0, -1.0): "EL",
    (-1.0, -0.5): "VL",
    (-1.0, 0): "L",
    (-1.0, 0.5): "~VH",
    (-1.0, 1.0): "A",
    (-0.5, 0.0): "SL",
    (-0.5, 0.5): "N",
    (-0.5, 1.0): "~VL",
    (0.0, 0.0): "VN",
    (0.0, 0.5): "SH",
    (0.0, 1.0): "H",
    (0.5, 1.0): "VH",
}

# for intervention_idx in range(len(labels)):
strengths = (
    InterventionStrength.WEAK,
    # InterventionStrength.MODERATE,
    InterventionStrength.STRONG,
)
# intervention_idxes = [0,1,2,5,6]
intervention_idxes = [2, 5, 6, 0, 1]
used_features = [True, True, True, False, False, True, True, True]

H = 1.75 * len(intervention_idxes)
W = (H / len(intervention_idxes)) * 3.25
print(W)
# H = W * (2/3) * len(intervention_idxes)
fig, grid_axes = plt.subplots(
    nrows=len(intervention_idxes),
    ncols=3,
    figsize=(W, H),
    constrained_layout=True,
    gridspec_kw={
        "width_ratios": [3, np.sum(used_features), len(strengths)],
        "height_ratios": [2, 3, 2, 4, 3],
        "wspace": 0.01,
        "hspace": 0.15,
    },
)

fig_dir = Path("../reports/thesis/results/figures")


for row_idx, intervention_idx in enumerate(intervention_idxes):
    target_idx = 7

    tree_fig, tree_axes = plt.subplots(nrows=3, figsize=(5, 8))

    rules = dict()
    rule_frequency = dict()
    axes = grid_axes[row_idx]

    for i, intervention in enumerate(strengths):
        if intervention == InterventionStrength.NULL:
            continue

        Y = diffs[intervention_idx][intervention]

        model = DecisionTreeRegressor(max_depth=4)
        # fit_X0 = X0.copy()
        # fit_X0[:, 2] = 0
        model.fit(X0, Y)
        plot_tree(model, feature_names=labels, ax=tree_axes[i])

        q = np.percentile(Y, 75)
        # q = np.percentile(Y, 55)

        _rules = extract_rules(model, labels, ge=q)
        rules[intervention] = [rule for _, rule in _rules]
        for freq, rule in _rules:
            if rule not in rule_frequency:
                rule_frequency[rule] = {}
            rule_frequency[rule][intervention] = freq

    # Figure out common rules
    all_rules = dict()
    for intervention, int_rules in rules.items():
        for rule in int_rules:
            # rule = tuple(sorted(rule, key=lambda x: x.feature_idx))
            all_rules[rule] = all_rules.get(rule, []) + [intervention]

    # For each rule, create:
    # 1. Heatmap fill (interval midpoint of each feature)
    # 2. Heatmap annotation (interval)
    # 3. Indicator matrix of which intervention strengths are applicable
    heatmap_vals = np.zeros((len(all_rules), len(labels)), dtype=np.float64)
    heatmap_annots = [["" for _ in range(len(labels))] for _ in range(len(all_rules))]
    mask = np.ones_like(heatmap_vals, dtype=np.bool)
    for rule_idx, rule in enumerate(all_rules):
        for rule_node in rule:
            heatmap_vals[rule_idx, rule_node.feature_idx] = (
                rule_node.interval.lower + rule_node.interval.upper
            ) / 2
            heatmap_annots[rule_idx][rule_node.feature_idx] = ANNOTATIONS[
                (rule_node.interval.lower, rule_node.interval.upper)
            ]
            mask[rule_idx, rule_node.feature_idx] = False
    heatmap_annots = np.asarray(heatmap_annots)

    # For each rule and strength, count how many individuals in the top quartile satisfy
    # Note: Mask when rule not applicable
    prevalence = np.zeros((len(all_rules), len(strengths)), dtype=np.float64)
    mask_prevalence = np.ones_like(prevalence, dtype=np.bool)
    for rule_idx, (rule, rule_strengths) in enumerate(all_rules.items()):
        for strength_idx, strength in enumerate(strengths):
            if strength not in rule_strengths:
                continue
            mask_prevalence[rule_idx, strength_idx] = False
            Y = diffs[intervention_idx][strength]
            q = np.percentile(Y, 75)
            X0_top = X0[q <= Y]
            X0_satisfy = X0_top.copy()
            for rule_node in rule:
                lower = (
                    rule_node.interval.lower if rule_node.interval.lower > -1 else -1.1
                )
                upper = (
                    rule_node.interval.upper if rule_node.interval.upper < 1 else 1.1
                )
                satisfies = (lower < X0_satisfy[:, rule_node.feature_idx]) & (
                    X0_satisfy[:, rule_node.feature_idx] <= upper
                )
                X0_satisfy = X0_satisfy[satisfies]
            prevalence[rule_idx, strength_idx] = X0_satisfy.shape[0] / X0_top.shape[0]

    # Only keep rules where at least one intervention has prevalence >= 15%
    keep_rows = (prevalence > 0.15).any(axis=1)
    heatmap_vals = heatmap_vals[keep_rows]
    heatmap_annots = heatmap_annots[keep_rows]
    mask = mask[keep_rows]
    prevalence = prevalence[keep_rows]
    mask_prevalence = mask_prevalence[keep_rows]

    # Remove any columns which don't feature in any rules
    # used_features = ~mask.all(axis=0)

    heatmap_vals = heatmap_vals[:, used_features]
    heatmap_annots = heatmap_annots[:, used_features]
    mask = mask[:, used_features]
    used_labels = labels[used_features]

    # == Determine column ordering
    # First identify any columns which are all -ve or +ve, i.e., necessary. Put first.
    # Put empty cols last
    necessary_cond = np.abs(heatmap_vals.mean(axis=0)) > 0.4
    empty_cond = mask.all(axis=0)
    necessary_cond_cols = np.argwhere(necessary_cond).flatten()
    empty_cond_cols = np.argwhere(empty_cond).flatten()
    unnecessary_cond_cols = np.argwhere(~necessary_cond & ~empty_cond).flatten()
    # unnecessary_cond_cols = np.argwhere(~necessary_cond).flatten()

    # Then re-order remaining cols by dendrogram
    cluster_cols = heatmap_vals[:, ~necessary_cond & ~empty_cond]
    # cluster_cols = heatmap_vals[:, ~necessary_cond]
    if cluster_cols.shape[-1] < 2:
        remaining_col_idxes = unnecessary_cond_cols
    else:
        clust = sns.clustermap(cluster_cols, row_cluster=False)
        plt.close(clust.fig)
        remaining_col_idxes = unnecessary_cond_cols[
            np.asarray(clust.dendrogram_col.reordered_ind)
        ]

    # Re-order the matrix
    col_idxes = np.concat((necessary_cond_cols, remaining_col_idxes, empty_cond_cols))
    # col_idxes = np.concat((necessary_cond_cols, remaining_col_idxes))
    # heatmap_vals = heatmap_vals[:, col_idxes]

    # And re-order rows by dendrogram
    clust = sns.clustermap(heatmap_vals[:, col_idxes], col_cluster=False)
    plt.close(clust.fig)
    row_idxes = clust.dendrogram_row.reordered_ind

    # Re-order rows
    weights = 2 ** np.arange(mask_prevalence.shape[1] - 1, -1, -1)
    values = mask_prevalence @ weights
    row_idxes = np.argsort(values)

    # Finally, update all of the matrices with re-ordering
    heatmap_vals = heatmap_vals[row_idxes][:, col_idxes]
    heatmap_annots = heatmap_annots[row_idxes][:, col_idxes]
    mask = mask[row_idxes][:, col_idxes]
    used_labels = used_labels[col_idxes]
    prevalence = prevalence[row_idxes]
    mask_prevalence = mask_prevalence[row_idxes]

    colours = ["#DDAA33", "#BB5566"]
    for strength, colour in zip(strengths[-1:], colours[-1:], strict=True):
        Y = diffs[intervention_idx][strength]
        q = np.percentile(Y, 75)
        Y_low = Y[q > Y]
        Y_high = Y[q <= Y]
        sns.kdeplot(Y, clip=(None, q), color=colour, fill=True, alpha=0.3, ax=axes[0])
        sns.kdeplot(Y, clip=(q, None), color=colour, fill=True, alpha=0.6, ax=axes[0])
    axes[0].set_box_aspect(heatmap_vals.shape[0] / 3)

    sns.heatmap(
        heatmap_vals,
        vmin=-0.7,
        vmax=0.7,
        center=0,
        annot=heatmap_annots,
        fmt="",
        annot_kws={"fontsize": 10},
        mask=mask,
        linewidths=1,
        cmap=DIVERGING_CMAP,
        ax=axes[1],
        square=True,
        cbar=False,
    )

    sns.heatmap(
        prevalence,
        annot=True,
        fmt=".2f",
        annot_kws={"fontsize": 10},
        vmin=0,
        vmax=1,
        cmap="crest",
        cbar=False,
        linewidths=1,
        ax=axes[2],
        square=True,
        mask=mask_prevalence,
    )

    # axes[0].set_ylabel(labels[intervention_idx])
    # axes[0].set_title("Intervention effect")#, pad=3)
    axes[0].spines.top.set_visible(False)
    axes[0].spines.right.set_visible(False)

    axes[1].set_yticks([])
    # axes[1].set_ylabel("Kinds")
    display_labels = used_labels[~mask.all(axis=0)]
    axes[1].set_xticks(
        np.arange(len(display_labels)) + 0.5,
        display_labels,
        rotation=35,
        horizontalalignment="right",
    )
    # axes[1].set_title("Belief/Attitude")#, pad=3)

    axes[2].set_yticks([])
    axes[2].set_xticks(
        np.arange(len(strengths)) + 0.5,
        [s.title() for s in strengths],
        rotation=35,
        horizontalalignment="right",
    )
    # axes[2].set_title("Prevalence")#, pad=3)

    axes[2].set_ylabel(labels[intervention_idx], rotation=0, labelpad=40)
    axes[2].yaxis.set_label_position("right")
    # axes[1].set_title(labels[intervention_idx])

    tree_fig.savefig(
        fig_dir / f"tree_{labels[intervention_idx].replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )

    plt.close(tree_fig)

grid_axes[0, 0].set_title("Intervention effect", pad=10)
grid_axes[0, 1].set_title("High-effect personas", pad=10)
grid_axes[0, 2].set_title("Prevalence", pad=10)

fig.savefig(
    fig_dir / f"rules_75_{labels[target_idx].replace(' ', '_')}.pdf",
    bbox_inches="tight",
)
fig.savefig(
    fig_dir / f"rules_75_{labels[target_idx].replace(' ', '_')}.png",
    bbox_inches="tight",
)
plt.close(fig)

In [ ]:
subset_labels = labels[[True, True, False, True, True, True, True, True]]
subset_X0 = X0[:, [True, True, False, True, True, True, True, True]]

Verify that CC worry Low is actually always required.

In [ ]:
# Intervention on belief in climate change
# Expect that effective interventions don't have high or low belief, or high or low
# policy
Y = diffs[0]["weak"]

In [ ]:
q = np.percentile(Y, 75)
q

In general population most people believe in climate change. Among those for whom 'Belief CC' is an effective intervention, far fewer believe in climate change. For these people, the majority state is _not_ believing in climate change. However, this is only distinctive in comparison with the broader population.

In [ ]:
fig, ax = plt.subplots()
sns.kdeplot(X0[:, 0], fill=True, alpha=0.5, ax=ax)
sns.kdeplot(X0[q <= Y, 0], fill=True, alpha=0.5, ax=ax)

In [ ]:
fig, ax = plt.subplots()
sns.kdeplot(X0[:, 7], fill=True, alpha=0.5, ax=ax)
sns.kdeplot(X0[q <= Y, 7], fill=True, alpha=0.5, ax=ax)

In [ ]:
fig, ax = plt.subplots()
sns.kdeplot(X0[:, 2], fill=True, alpha=0.5, ax=ax)
sns.kdeplot(X0[q <= Y, 2], fill=True, alpha=0.5, ax=ax)

In [ ]:
labels

In [ ]:
labels

In [ ]:
X0[:, 2][q <= Y]

In [ ]:
X0

In [ ]:
all_rules
# set(tuple(rule) for rule in rules["moderate"])

In [ ]:
plot_tree(model)

In [ ]:
fig, axes = plt.subplots(
    ncols=2, sharey=True, constrained_layout=True, gridspec_kw={"width_ratios": [8, 3]}
)

data1 = np.random.normal(size=(4, 8))
data2 = np.random.binomial(1, 0.5, size=(4, 3))

annot_labels = [
    [f"$({float(col):.1f},{float(col) + 0.15:.1f}]$" for col in row] for row in data1
]

sns.heatmap(
    data1,
    vmin=-2,
    vmax=2,
    center=0,
    annot=annot_labels,
    fmt="",
    annot_kws={"fontsize": 6.5},
    linewidths=1,
    cmap=DIVERGING_CMAP,
    ax=axes[0],
    square=True,
    cbar=False,
)
sns.heatmap(
    data2,
    vmin=0,
    vmax=1,
    linewidths=1,
    ax=axes[1],
    square=True,
    cbar=False,
    mask=data2 == 1,
)

axes[0].set_yticks([])
axes[0].set_xticks(
    np.arange(len(labels)) + 0.5, labels, rotation=30, horizontalalignment="right"
)
axes[1].set_xticks(
    np.arange(3) + 0.5,
    ["Weak", "Moderate", "Strong"],
    rotation=30,
    horizontalalignment="right",
)
axes[0].xaxis.set_label_position("top")
axes[1].xaxis.set_label_position("top")
axes[0].set_xlabel("Belief/Attitude", labelpad=10)
axes[1].set_xlabel("Intervention strength", labelpad=10)

# axes[0].imshow(data1, aspect="equal", cmap=DIVERGING_CMAP)
# axes[1].imshow(data2, aspect="equal", cmap="Greys")

## Comparing regression and tree models

In [ ]:
from sklearn.model_selection import RepeatedKFold, cross_validate

$R^2$ score

In [ ]:
lasso = Pipeline([("poly", PolynomialFeatures()), ("lasso", LassoCV(cv=10))])
tree = DecisionTreeRegressor(max_depth=3)

cv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=RANDOM_SEED)

lasso_scores = cross_validate(lasso, X0, Y, cv=cv, scoring="r2")

tree_scores = cross_validate(tree, X0, Y, cv=cv, scoring="r2")

In [ ]:
(
    lasso_scores["test_score"].mean(),
    tree_scores["test_score"].mean(),
)  # , rf_scores["test_score"].mean()

How well does each model recover the top-percentile (most effective) intervention individuals?

In [ ]:
true_top_10_pct = np.argwhere(np.percentile(Y, q=90) <= Y).flatten()

lasso_pred = lasso.fit(X0, Y).predict(X0)
tree_pred = tree.fit(X0, Y).predict(X0)

lasso_top_10_pct = np.argwhere(lasso_pred >= np.percentile(lasso_pred, q=90)).flatten()
tree_top_10_pct = np.argwhere(tree_pred >= np.percentile(tree_pred, q=90)).flatten()

In [ ]:
# How many true top-percentile respondents are recovered?
recovery_lasso = (
    np.intersect1d(true_top_10_pct, lasso_top_10_pct).size / true_top_10_pct.size
)
recovery_tree = (
    np.intersect1d(true_top_10_pct, tree_top_10_pct).size / true_top_10_pct.size
)

In [ ]:
recovery_lasso, recovery_tree